# 01b — Merge cleaned acquisitions and run FTH

Reload the separately cleaned output for every acquisition ID, average the selected IDs by polarization, then center the merged holograms, run FTH, and save one combined workflow data file.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

BASEFOLDER = Path.cwd().resolve()
sys.path.insert(0, str(BASEFOLDER / "library"))
import CCI_core as cci
import fthcore as fth
import fth_phase_workflow as wf
import helper_functions as helper
import interactive
import reconstruct_rb as rec
from mask_store import MaskStore

%matplotlib widget
print("Base folder:", BASEFOLDER)

## Configuration

In [ ]:
USER = "rb"
CLEANED_FOLDER = BASEFOLDER / "processed" / "cleaned_acquisitions"

# Current acquisition lists, center, and ROI from the newer working notebook.
PLUS_IMAGE_IDS = [569, 571, 605, 607, 609] + list(range(575, 602, 2))
MINUS_IMAGE_IDS = [570, 572, 606, 608, 610] + list(range(576, 603, 2))


## Reload separate acquisitions, then merge them

In [ ]:
# Load every cleaned acquisition separately. Each file contains the combined
# detector + beamstop mask saved by notebook 01a.
plus_images = []
minus_images = []
plus_masks = []
minus_masks = []
plus_files = []
minus_files = []
thresholds = []
dark_ids = None
energy = None
ccd_dist = None
px_size = None

for polarization, image_ids, images, masks, files in (
    ("+", PLUS_IMAGE_IDS, plus_images, plus_masks, plus_files),
    ("-", MINUS_IMAGE_IDS, minus_images, minus_masks, minus_files),
):
    for image_id in image_ids:
        input_file = CLEANED_FOLDER / f"cleaned_ImId_{image_id:04d}_{USER}.npz"
        with np.load(input_file, allow_pickle=False) as saved:
            if int(saved["image_id"]) != image_id:
                raise ValueError(f"Acquisition metadata does not match {input_file}")
            images.append(np.asarray(saved["image"], dtype=float))
            masks.append(np.asarray(saved["mask_pixel"], dtype=np.uint8))
            thresholds.append(float(saved["threshold"]))
            current_dark_ids = saved["dark_ids"].astype(int).tolist()
            current_setup = (
                float(saved["energy_eV"]),
                float(saved["ccd_dist_m"]),
                float(saved["px_size_m"]),
            )
        if dark_ids is None:
            dark_ids = current_dark_ids
            energy, ccd_dist, px_size = current_setup
        elif current_setup != (energy, ccd_dist, px_size):
            raise ValueError("Selected acquisitions do not share detector geometry")
        files.append(input_file)
        print(f"Loaded {polarization} ID {image_id}: {input_file}")


def average_valid_pixels(images, masks):
    image_stack = np.stack(images).astype(float)
    mask_stack = np.stack(masks).astype(bool)
    valid_stack = (~mask_stack) & np.isfinite(image_stack)
    summed = np.sum(np.where(valid_stack, image_stack, 0.0), axis=0)
    counts = np.sum(valid_stack, axis=0)
    average = np.divide(summed, counts, out=np.zeros_like(summed), where=counts > 0)
    missing = (counts == 0).astype(np.uint8)
    return average, missing, counts


# This is the first point where separate acquisition IDs are merged.
pos_raw, mask_pixel_pos_raw, source_count_pos = average_valid_pixels(
    plus_images, plus_masks
)
neg_raw, mask_pixel_neg_raw, source_count_neg = average_valid_pixels(
    minus_images, minus_masks
)
pos_raw = np.asarray(pos_raw, dtype=float)
neg_raw = np.asarray(neg_raw, dtype=float)
positive_ids = [int(image_id) for image_id in PLUS_IMAGE_IDS]
negative_ids = [int(image_id) for image_id in MINUS_IMAGE_IDS]
if pos_raw.shape != neg_raw.shape or pos_raw.ndim != 2:
    raise ValueError(f"Expected matching 2-D averages, got {pos_raw.shape} and {neg_raw.shape}")

im_id, topo_id = positive_ids[0], negative_ids[0]
folder_general = Path(helper.create_folder(BASEFOLDER / "processed"))
folder_logs = Path(helper.create_folder(folder_general / "Logs"))
DATA_H5 = folder_logs / f"data_recon_ImId_{im_id:04d}_{USER}.hdf5"
experimental_setup = {
    "ccd_dist": ccd_dist, "px_size": px_size, "binning": 1,
    "oversaturation": 60e3, "energy": energy,
    "lambda": helper.photon_energy_wavelength(energy, input_unit="eV"),
}

# The nested structure below is only the established HDF5 exchange format;
# user-editable configuration remains in the simple variables above.
data = {
    "workflow": "FTH_from_cleaned_3d_averages",
    "user": USER,
    "data_file": str(DATA_H5),
    "experimental_setup": experimental_setup,
    "positive_label": "+",
    "reference_label": "-",
    "hologram_labels": ["+", "-"],
    "preprocessed_files": {
        "+": [str(path) for path in plus_files],
        "-": [str(path) for path in minus_files],
    },
    "intensity_thresholds": thresholds,
    "holo": {
        "+": {"id": positive_ids, "dark_id": dark_ids, "image": pos_raw},
        "-": {"id": negative_ids, "dark_id": dark_ids, "image": neg_raw},
    },
}
print("Merged shapes:", pos_raw.shape, neg_raw.shape)
print("Output HDF5:", DATA_H5)


## Choose the detector center

In [ ]:
# Edit the starting center here, then adjust it in the widget.
CENTER_START = [997, 1040]

center_widget = interactive.InteractiveCenter(
    neg_raw, c0=CENTER_START[0], c1=CENTER_START[1]
)
# Adjust the widget before running the next cell.

In [ ]:
center = [center_widget.c0, center_widget.c1]
data["center"] = center
data = wf.define_centered_holograms(data, cci)
print("Center:", center)

## Load and center the detector mask

In [ ]:
# Center the per-polarization masks using the same center as the holograms.
mask_pixel_pos = (
    wf.center_image(mask_pixel_pos_raw, center, cci) > 0.5
).astype(np.uint8)
mask_pixel_neg = (
    wf.center_image(mask_pixel_neg_raw, center, cci) > 0.5
).astype(np.uint8)

# A difference hologram is unusable where either polarization is missing.
mask_pixel = np.maximum(mask_pixel_pos, mask_pixel_neg).astype(np.uint8)
data["mask_pixel"] = mask_pixel
data["mask_pixel_raw_by_label"] = {
    "+": mask_pixel_pos_raw,
    "-": mask_pixel_neg_raw,
}
data["holo"]["+"]["mask_pixel_raw"] = mask_pixel_pos_raw
data["holo"]["-"]["mask_pixel_raw"] = mask_pixel_neg_raw
data["holo"]["+"]["mask_pixel_c"] = mask_pixel_pos
data["holo"]["-"]["mask_pixel_c"] = mask_pixel_neg
data["holo"]["+"]["source_count"] = source_count_pos
data["holo"]["-"]["source_count"] = source_count_neg
print("Final combined mask_pixel pixels:", int(mask_pixel.sum()))


## Construct and inspect the FTH hologram

In [ ]:
# FTH masking parameters are here because the masks are constructed below.
MASK_PIXEL_DILATION = 3
MASK_PIXEL_BLUR_SIGMA = 3.0
BUTTERWORTH_RADIUS = 35
BUTTERWORTH_ORDER = 4

shape = pos_raw.shape

# 1) Slightly enlarge binary mask_pixel, then blur its edge.
mask_pixel_fth = wf.smooth_binary_mask(
    mask_pixel.astype(float),
    MASK_PIXEL_DILATION,
    MASK_PIXEL_BLUR_SIGMA,
)

# 2) Make the separate smooth Butterworth exclusion at the Fourier origin.
mask_beamstop_smooth = wf.butterworth_disk_mask(
    shape,
    BUTTERWORTH_RADIUS,
    BUTTERWORTH_ORDER,
)

# Convert both exclusion masks to transmission and combine them.
mask_multiplier = (1 - mask_pixel_fth) * (1 - mask_beamstop_smooth)

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for axis, shown_mask, title in zip(
    axes,
    (mask_pixel, mask_pixel_fth, mask_beamstop_smooth, mask_multiplier),
    ("binary mask_pixel", "dilated + blurred mask_pixel",
     "Butterworth exclusion mask", "final FTH transmission"),
):
    image = axis.imshow(shown_mask, cmap="gray", vmin=0, vmax=1)
    axis.set_title(title)
    axis.set_axis_off()
    fig.colorbar(image, ax=axis, fraction=0.046)
plt.tight_layout()
plt.show()
pos = np.asarray(data["holo"]["+"]["image_c"], dtype=float)
neg = np.asarray(data["holo"]["-"]["image_c"], dtype=float)
# Normalize polarizations only in the selected region and outside the full mask_pixel.
POLARIZATION_FIT_ROWS = slice(500, -500)
POLARIZATION_FIT_COLUMNS = slice(500, -500)
POLARIZATION_FIT_PERCENTILES = (2, 98)

fit_region = np.zeros(pos.shape, dtype=bool)
fit_region[POLARIZATION_FIT_ROWS, POLARIZATION_FIT_COLUMNS] = True
fit_pixels = (mask_pixel == 0) & fit_region & np.isfinite(pos) & np.isfinite(neg)
low, high = np.percentile(pos[fit_pixels], POLARIZATION_FIT_PERCENTILES)
fit_pixels &= (pos >= low) & (pos <= high)
factor, offset = cci.dyn_factor(
    pos[fit_pixels], neg[fit_pixels],
    method="correlation", verbose=False, plot=False,
)

plot_step = max(1, fit_pixels.sum() // 7000)
plot_x = pos[fit_pixels][::plot_step]
plot_y = neg[fit_pixels][::plot_step]
line_x = np.linspace(plot_x.min(), plot_x.max(), 300)
fig, axis = plt.subplots(figsize=(6, 5))
axis.scatter(plot_x, plot_y, s=3, alpha=0.15, label="valid fit pixels")
axis.plot(line_x, line_x / factor - offset, color="red", linewidth=2,
          label=f"fit: x/{factor:.5g} - {offset:.5g}")
axis.set_title("Polarization normalization outside mask_pixel")
axis.set_xlabel("+ intensity")
axis.set_ylabel("- intensity")
axis.legend()
axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()
pos_normalized = pos / factor
holo_unmasked = pos_normalized - neg - offset
holo_masked = holo_unmasked * mask_multiplier
data["factor"], data["offset"] = float(factor), float(offset)
data["mask_beamstop_smooth_recipe"] = {"radius": BUTTERWORTH_RADIUS, "order": BUTTERWORTH_ORDER}
data["mask_pixel_fth_recipe"] = {"dilation_pixels": MASK_PIXEL_DILATION, "sigma": MASK_PIXEL_BLUR_SIGMA}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, image, title in zip(axes, (pos, neg, holo_masked), ("+ average", "- average", "masked difference")):
    vmin, vmax = wf.finite_percentile_limits(image, (1, 99.9))
    axis.imshow(image, vmin=vmin, vmax=vmax, cmap="viridis")
    axis.set_title(title)
    axis.set_axis_off()
plt.tight_layout(); plt.show()

## Focus and crop the FTH reconstruction

In [ ]:
# Edit the reconstruction ROI here, immediately before focusing.
ROI = [315, 605, 200, 490]

roi_s = np.s_[ROI[0]:ROI[1], ROI[2]:ROI[3]]
prop_dist, phase, dx, dy = 0.0, 0.0, 0.0, 0.0
focus_sliders = rec.focusCDI(
    holo_masked, np.zeros_like(holo_masked), roi_s, mask=1,
    phase=phase, prop_dist=prop_dist, dx=dx, dy=dy,
    experimental_setup=experimental_setup, operation="-",
    max_prop_dist=30, scale=(2, 98),
)
slider_prop, slider_phase, slider_dx, slider_dy = focus_sliders[:4]

## Apply the selected focus and save

In [ ]:
prop_dist = float(slider_prop.value)
phase = float(slider_phase.value)
dx = float(slider_dx.value)
dy = float(slider_dy.value)
reconstruction = wf.fth_reconstruct(
    holo_masked, experimental_setup, fth,
    prop_dist=prop_dist, phase=phase, dx=dx, dy=dy,
)
data["focus_fth"] = {
    "prop_dist": prop_dist, "prop_dist_unit": "um",
    "phase": phase, "dx": dx, "dy": dy,
    "roi": ROI, "operation": "-",
}
data["recon"] = reconstruction[roi_s]
png_name = folder_general / f"FTH_recon_ImId_{im_id:04d}_{USER}.png"
fig, axis = plt.subplots(figsize=(5, 5))
shown = np.real(data["recon"])
vmin, vmax = wf.finite_percentile_limits(shown)
axis.imshow(shown, vmin=vmin, vmax=vmax, cmap="gray")
axis.set_title(f"Focused FTH — averaged IDs beginning at {im_id}")
axis.set_axis_off()
fig.savefig(png_name, bbox_inches="tight", dpi=200)
plt.show()
data["fth_png"] = str(png_name)
wf.save_data_dict(data, DATA_H5, overwrite=True)
print("Saved HDF5:", DATA_H5)
print("Saved PNG:", png_name)

In [ ]:
print("plus im_ids:", positive_ids)
print("minus im_ids:", negative_ids)
print("dark_ids:", dark_ids)


## Simple names for interactive inspection

These match the names in `01_FTH.ipynb`, so plotting works the same way in both workflows.


In [ ]:
# The same conventional NumPy names used in the raw FTH notebook.
pos = np.asarray(pos, dtype=float)
neg = np.asarray(neg, dtype=float)
holo = np.asarray(holo_masked, dtype=float)
recon = np.asarray(data["recon"])


def show_image(image, title="image", cmap="viridis", percentiles=(1, 99.9)):
    image = np.asarray(image)
    values = np.real(image) if np.iscomplexobj(image) else image.astype(float)
    finite = values[np.isfinite(values)]
    vmin, vmax = np.percentile(finite, percentiles)
    fig, axis = plt.subplots(figsize=(6, 5))
    shown = axis.imshow(values, cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set_title(title)
    axis.set_axis_off()
    fig.colorbar(shown, ax=axis)
    plt.tight_layout()
    plt.show()
    return fig, axis


print("Easy NumPy names: pos_raw, neg_raw, pos, neg, mask_pixel, holo, recon")
print("Example: show_image(pos, 'positive image')")
print("Example: show_image(holo, 'FTH hologram')")
print("Example: show_image(recon, 'FTH reconstruction', cmap='gray')")
